# Cityguard — dziury i pęknięcia (YOLO12)

Fine-tuning `yolo12n.pt` na Kaggle `aliabdelmenam/rdd-2022`. Wynik: `best_road.pt`.

Śmieci trenuj osobno w `ml/train_litter.ipynb`, najlepiej na drugim koncie Colab.

Środowisko: **GPU T4**. Przy pierwszym pobraniu wgraj `kaggle.json` (Kaggle → Settings → API → Create New Token). RDD2022 ściąga się w całości; do treningu idzie próbka z kamer na pojeździe.


In [ ]:
!pip -q install -U ultralytics kagglehub


In [ ]:
import torch

!nvidia-smi
if not torch.cuda.is_available():
    raise SystemExit("Włącz GPU: Środowisko wykonawcze → Zmień typ środowiska → T4")
print("GPU:", torch.cuda.get_device_name(0))


## Konfiguracja

`MAX_IMAGES = 0` bierze wszystkie klatki po filtrze krajów. Na darmowym T4 zostaw 2000.

`ROAD_COUNTRIES` filtruje prefiks nazwy pliku (`Japan_008913.jpg`), nie folder. Dron (`China_Drone_`) i Street View odpadają.


In [ ]:
DATASET = "aliabdelmenam/rdd-2022"
ROAD_COUNTRIES = ["Japan", "Czech", "India", "Norway"]
MAX_IMAGES = 2000  # 0 = cały przefiltrowany zbiór
EPOCHS = 25
IMGSZ = 640
BATCH = 16
CONF = 0.35
SAVE_TO_DRIVE = False


In [ ]:

import json
import random
import shutil
from pathlib import Path

import kagglehub
import yaml
from PIL import Image
from ultralytics import YOLO

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def peek(root: Path, limit: int = 25) -> None:
    shown = 0
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        print(" ", path.relative_to(root))
        shown += 1
        if shown >= limit:
            print("  ...")
            return


def download(slug: str) -> Path:
    print("Pobieram", slug)
    root = Path(kagglehub.dataset_download(slug))
    print("Kaggle:", root)
    peek(root)
    return root


def link_split(pairs: list[tuple[Path, str]], dest: Path, names: list[str]) -> Path:
    if dest.exists():
        shutil.rmtree(dest)
    for split in ("train", "val"):
        (dest / "images" / split).mkdir(parents=True)
        (dest / "labels" / split).mkdir(parents=True)
    random.Random(42).shuffle(pairs)
    val_count = max(1, int(len(pairs) * 0.1))
    if len(pairs) < 2:
        raise RuntimeError(f"Za mało obrazów z etykietą: {len(pairs)}")
    for index, (image, label) in enumerate(pairs):
        split = "val" if index < val_count else "train"
        stem = f"{index:05d}_{image.stem}"
        target = dest / "images" / split / f"{stem}{image.suffix.lower()}"
        target.symlink_to(image.resolve())
        (dest / "labels" / split / f"{stem}.txt").write_text(label, encoding="utf-8")
    yaml_path = dest / "cityguard.yaml"
    payload = {
        "train": str((dest / "images" / "train").resolve()),
        "val": str((dest / "images" / "val").resolve()),
        "names": {i: name for i, name in enumerate(names)},
    }
    yaml_path.write_text(yaml.safe_dump(payload, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print(f"{dest.name}: {len(pairs) - val_count} train / {val_count} val, klasy {names}")
    return yaml_path


def backup_epoch(weight_name: str):
    def _hook(trainer):
        if not SAVE_TO_DRIVE:
            return
        best = Path(trainer.save_dir) / "weights" / "best.pt"
        if not best.exists():
            return
        target = Path("/content/drive/MyDrive/cityguard")
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy(best, target / weight_name)
        print(f"Drive: {weight_name} po epoce {trainer.epoch + 1}")

    return _hook


def fit_yolo(data_yaml: Path, run_name: str, weight_name: str) -> Path:
    import torch

    def fit(current_batch: int):
        model = YOLO("yolo12n.pt")
        model.add_callback("on_fit_epoch_end", backup_epoch(weight_name))
        model.train(
            data=str(data_yaml),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=current_batch,
            device=0,
            project="/content/runs",
            name=run_name,
            exist_ok=True,
            patience=8,
            workers=2,
            plots=True,
            save_period=5,
        )
        return model

    try:
        model = fit(BATCH)
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        print("CUDA OOM. Powtarzam z batch=8.")
        torch.cuda.empty_cache()
        model = fit(8)

    best = Path("/content/runs") / run_name / "weights" / "best.pt"
    out = Path("/content") / weight_name
    shutil.copy(best, out)
    classes = {"task": run_name, "conf": CONF, "names": {str(key): value for key, value in model.names.items()}}
    Path(f"/content/classes_{run_name}.json").write_text(json.dumps(classes, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Wagi:", out)
    return out


import xml.etree.ElementTree as ET

ROAD_LABELS = {
    "D00": "crack",
    "D01": "crack",
    "D10": "crack",
    "D11": "crack",
    "D20": "crack",
    "D40": "pothole",
    "D43": "road_marking",
    "D44": "road_marking",
    "D50": "manhole",
}


def voc_label(xml_path: Path, class_to_id: dict[str, int]) -> str:
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    width = float(size.findtext("width") or 0) if size is not None else 0
    height = float(size.findtext("height") or 0) if size is not None else 0
    if width <= 1 or height <= 1:
        image = next((xml_path.with_suffix(ext) for ext in IMAGE_EXT if xml_path.with_suffix(ext).exists()), None)
        if image is None:
            return ""
        with Image.open(image) as handle:
            width, height = handle.size
    lines = []
    for obj in root.findall("object"):
        raw_name = (obj.findtext("name") or "").strip()
        mapped = ROAD_LABELS.get(raw_name, raw_name)
        if mapped not in class_to_id:
            continue
        box = obj.find("bndbox")
        if box is None:
            continue
        xmin = max(0.0, float(box.findtext("xmin") or 0))
        ymin = max(0.0, float(box.findtext("ymin") or 0))
        xmax = min(width, float(box.findtext("xmax") or 0))
        ymax = min(height, float(box.findtext("ymax") or 0))
        if xmax <= xmin or ymax <= ymin:
            continue
        lines.append(
            f"{class_to_id[mapped]} {(xmin + xmax) / 2 / width:.6f} {(ymin + ymax) / 2 / height:.6f} "
            f"{(xmax - xmin) / width:.6f} {(ymax - ymin) / height:.6f}"
        )
    return "\n".join(lines)


def country_ok(path: Path, wanted: set[str]) -> bool:
    parts = {part.lower() for part in path.parts}
    if "test" in parts:
        return False
    if not wanted:
        return True
    stem = path.stem.lower()
    for name in wanted:
        token = name.lower().replace(" ", "_")
        if stem.startswith(token + "_") or token in parts:
            return True
    return False


def dataset_names(root: Path) -> list[str] | None:
    for path in root.rglob("*"):
        if path.suffix.lower() not in {".yaml", ".yml"}:
            continue
        try:
            data = yaml.safe_load(path.read_text(encoding="utf-8", errors="ignore"))
        except Exception:
            continue
        names = data.get("names") if isinstance(data, dict) else None
        if isinstance(names, dict) and names:
            return [str(names[key]) for key in sorted(names, key=lambda item: int(item))]
        if isinstance(names, list) and names:
            return [str(name) for name in names]
    for path in root.rglob("*"):
        if path.name.lower() in {"classes.txt", "obj.names"}:
            lines = [line.strip() for line in path.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
            if lines:
                return lines
    return None


def canonical_name(name: str) -> str:
    key = " ".join(name.strip().lower().replace("-", " ").replace("_", " ").split())
    aliases = {
        "d00": "crack",
        "d01": "crack",
        "d10": "crack",
        "d11": "crack",
        "d20": "crack",
        "longitudinal crack": "crack",
        "transverse crack": "crack",
        "alligator crack": "crack",
        "crack": "crack",
        "d40": "pothole",
        "pothole": "pothole",
        "d43": "road_marking",
        "d44": "road_marking",
        "d50": "manhole",
    }
    return aliases.get(key, name.strip())


def yolo_rows(label_path: Path) -> list[tuple[int, float, float, float, float]]:
    rows = []
    for line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        try:
            cls = int(float(parts[0]))
            x, y, width, height = (float(value) for value in parts[1:5])
        except ValueError:
            return []
        rows.append((cls, x, y, width, height))
    return rows


def collect_yolo(root: Path, wanted: set[str]) -> list[tuple[Path, list[tuple[int, float, float, float, float]]]]:
    images: dict[str, Path] = {}
    for path in root.rglob("*"):
        if path.suffix.lower() in IMAGE_EXT:
            images.setdefault(path.stem, path)
    skip = {"train.txt", "valid.txt", "val.txt", "test.txt", "classes.txt", "obj.names", "obj.data"}
    found = []
    for label_path in root.rglob("*.txt"):
        if label_path.name.lower() in skip or not country_ok(label_path, wanted):
            continue
        image = images.get(label_path.stem)
        if image is None:
            continue
        rows = yolo_rows(label_path)
        if rows:
            found.append((image, rows))
    return found


def prepare_road(dest: Path) -> Path:
    raw = download(DATASET)
    wanted = {name.lower() for name in ROAD_COUNTRIES}
    found = collect_yolo(raw, wanted)
    if not found and wanted:
        print("Brak plików", sorted(wanted), "w nazwie (np. Japan_008913.jpg). Biorę wszystkie etykiety YOLO.")
        found = collect_yolo(raw, set())
    if found:
        source_names = dataset_names(raw) or []
        print("Klasy ze zbioru:", source_names or "(brak data.yaml, zostawiam class_N)")
        print("Par obraz+etykieta:", len(found))
        used = sorted({cls for _, rows in found for cls, *_rest in rows})
        id_to_name = {
            cls: canonical_name(source_names[cls]) if cls < len(source_names) else f"class_{cls}"
            for cls in used
        }
        present: list[str] = []
        for name in id_to_name.values():
            if name not in present:
                present.append(name)
        preferred = ["pothole", "crack", "road_marking", "manhole"]
        names = [name for name in preferred if name in present] + sorted(name for name in present if name not in preferred)
        name_to_new = {name: index for index, name in enumerate(names)}
        old_to_new = {cls: name_to_new[id_to_name[cls]] for cls in used}
        random.Random(42).shuffle(found)
        if MAX_IMAGES and len(found) > MAX_IMAGES:
            found = found[:MAX_IMAGES]
        labeled = []
        for image, rows in found:
            absolute = any(max(abs(x), abs(y), abs(w), abs(h)) > 1.5 for _cls, x, y, w, h in rows)
            width = height = 1.0
            if absolute:
                with Image.open(image) as handle:
                    width, height = handle.size
            lines = []
            for cls, x, y, w, h in rows:
                if cls not in old_to_new:
                    continue
                if absolute:
                    x, y, w, h = x / width, y / height, w / width, h / height
                lines.append(f"{old_to_new[cls]} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
            if lines:
                labeled.append((image, "\n".join(lines) + "\n"))
        if not labeled:
            raise FileNotFoundError("RDD2022: etykiety YOLO są puste")
        return link_split(labeled, dest, names)

    wanted_parts = {name.lower() for name in ROAD_COUNTRIES}
    pairs = []
    for xml_path in raw.rglob("*.xml"):
        if not country_ok(xml_path, wanted_parts):
            continue
        image = next((xml_path.with_suffix(ext) for ext in IMAGE_EXT if xml_path.with_suffix(ext).exists()), None)
        if image is not None:
            pairs.append((image, xml_path))
    if not pairs:
        raise FileNotFoundError("RDD2022: nie znalazłem par obraz+etykieta (ani YOLO txt, ani XML)")
    random.Random(42).shuffle(pairs)
    if MAX_IMAGES and len(pairs) > MAX_IMAGES:
        pairs = pairs[:MAX_IMAGES]
    present = []
    for _, xml_path in pairs:
        root = ET.parse(xml_path).getroot()
        for obj in root.findall("object"):
            raw_name = (obj.findtext("name") or "").strip()
            mapped = canonical_name(ROAD_LABELS.get(raw_name, raw_name))
            if mapped and mapped not in present:
                present.append(mapped)
    preferred = ["pothole", "crack", "road_marking", "manhole"]
    names = [name for name in preferred if name in present] + sorted(name for name in present if name not in preferred)
    class_to_id = {name: index for index, name in enumerate(names)}
    labeled = []
    for image, xml_path in pairs:
        label = voc_label(xml_path, class_to_id)
        if label:
            labeled.append((image, label + "\n"))
    return link_split(labeled, dest, names)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

fit_yolo(prepare_road(Path("/content/datasets/road")), "road", "best_road.pt")



## Pobranie wag

Rozpakuj ZIP do `ml/weights/`. Inferencja szuka `best_road.pt`.


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

bundle = Path("/content/cityguard_best_road.pt.zip")
names = ["best_road.pt", "classes_road.json"]
with zipfile.ZipFile(bundle, "w") as archive:
    found = False
    for name in names:
        path = Path("/content") / name
        if path.exists():
            archive.write(path, name)
            found = True
            print("pakuję", name)
        else:
            print("pomijam", name)
if not found:
    raise SystemExit("Nie ma jeszcze wag. Najpierw odpal trening.")
files.download(str(bundle))


## Podgląd na klatkach z demo

Ustaw `RUN_PREVIEW = True` i uruchom komórkę ręcznie. Próg: `CONF` (0.35).


In [ ]:
RUN_PREVIEW = False  # ustaw True, żeby wgrać klatki i narysować boxy

if not RUN_PREVIEW:
    print("Podgląd wyłączony. Ustaw RUN_PREVIEW = True i odpal tę komórkę ponownie.")
else:
    from google.colab import files
    from pathlib import Path
    from ultralytics import YOLO
    preview = Path("/content/preview_images")
    preview.mkdir(exist_ok=True)
    for name, blob in files.upload().items():
        (preview / name).write_bytes(blob)
    weight = Path("/content/best_road.pt")
    YOLO(str(weight)).predict(
        source=str(preview),
        conf=CONF,
        save=True,
        project="/content/preview_out",
        name="road",
        exist_ok=True,
    )
    print("Obrazki z boxami: /content/preview_out/road")
